# Training YOLOv8s Multi-Class di Colab T4 GPU — Tahap 2
Human Safety Monitoring System — INaAI 2026.

**PENTING — aktifkan GPU dulu:** menu `Runtime` -> `Change runtime type` -> Hardware accelerator = **T4 GPU** -> Save.

Parameter training (seed, augmentasi) dibuat **identik** dengan `train.py` lokal agar hasil reproducible.

## 1. Verifikasi GPU aktif

In [1]:
!nvidia-smi

Thu Jun 11 10:51:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Ultralytics

In [2]:
!pip install -q ultralytics==8.3.0
import ultralytics; ultralytics.checks()

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 52.9/118.9 GB disk)


## 3. Ambil dataset
Upload **`css-remapped.zip`** (156 MB) ke Google Drive Anda (taruh di root `My Drive`), lalu jalankan sel ini untuk mount + unzip.

Alternatif tanpa Drive: comment baris mount, uncomment `files.upload()` (lebih lambat & hilang jika sesi putus).

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!cp '/content/drive/MyDrive/css-remapped.zip' /content/css-remapped.zip

# --- Alternatif upload langsung (jika tidak pakai Drive) ---
# from google.colab import files
# files.upload()   # pilih css-remapped.zip
# !mv css-remapped.zip /content/css-remapped.zip

!unzip -q -o /content/css-remapped.zip -d /content/
!ls /content/construction-site-safety

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
README.dataset.txt  README.roboflow.txt  test  train  valid


## 4. Tulis data.yaml (path absolut Colab, 7 kelas)

In [4]:
from google.colab import drive
drive.mount('/content/drive')
!cp '/content/drive/MyDrive/css-remapped.zip' /content/css-remapped.zip

# --- Alternatif upload langsung (jika tidak pakai Drive) ---
# from google.colab import files
# files.upload()   # pilih css-remapped.zip
# !mv css-remapped.zip /content/css-remapped.zip

!unzip -q -o /content/css-remapped.zip -d /content/
!ls /content/construction-site-safety

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
README.dataset.txt  README.roboflow.txt  test  train  valid


In [5]:
data_yaml = '''path: /content/construction-site-safety
train: train/images
val:   valid/images
test:  test/images
names:
  0: Person
  1: Hardhat
  2: NO-Hardhat
  3: Safety-Vest
  4: NO-Safety-Vest
  5: Mask
  6: NO-Mask
'''
with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml)
print(data_yaml)

path: /content/construction-site-safety
train: train/images
val:   valid/images
test:  test/images
names:
  0: Person
  1: Hardhat
  2: NO-Hardhat
  3: Safety-Vest
  4: NO-Safety-Vest
  5: Mask
  6: NO-Mask



## 5. Training (params identik dengan train.py lokal)
T4 ~ 30-60 menit untuk 50 epoch. Ultralytics otomatis pakai GPU.

In [8]:
import numpy as np
print(np.__version__)

1.26.4


In [7]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor

In [ ]:
import random, numpy as np, torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True

model = YOLO('yolov8s.pt')
model.train(
    data='/content/data.yaml',
    epochs=50, imgsz=640, batch=16, seed=SEED,
    close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    scale=0.5, translate=0.1, fliplr=0.5, flipud=0.0,
    project='runs', name='yolov8s_ppe',
    device=0,
)
print('Selesai. best.pt di runs/yolov8s_ppe/weights/best.pt')

100%|██████████| 21.5M/21.5M [00:00<00:00, 169MB/s]


New https://pypi.org/project/ultralytics/8.4.65 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=runs, name=yolov8s_ppe, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=Fals

100%|██████████| 755k/755k [00:00<00:00, 15.5MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

100%|██████████| 6.25M/6.25M [00:00<00:00, 77.9MB/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      27/50      4.66G     0.9447     0.7818      1.146        201        640: 100%|██████████| 163/163 [00:53<00:00,  3.07it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.34it/s]

                   all        114        556      0.882      0.706      0.783      0.445



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      28/50      4.74G     0.9291     0.7645      1.134        247        640:  98%|█████████▊| 160/163 [00:52<00:00,  3.46it/s]

## 6. Eval di test set (gerbang go/no-go: mAP@0.5 >= 0.50)

In [ ]:
best = YOLO('runs/yolov8s_ppe/weights/best.pt')
m = best.val(data='/content/data.yaml', split='test', iou=0.5)
print('mAP@0.5      :', round(float(m.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(m.box.map), 4))
print('precision    :', round(float(m.box.mp), 4))
print('recall       :', round(float(m.box.mr), 4))

## 7. Download hasil ke lokal
Unduh `best.pt` -> taruh di `weights/best.pt` repo. Juga unduh kurva training untuk report.

In [ ]:
from google.colab import files
import shutil
shutil.copy('runs/yolov8s_ppe/weights/best.pt', '/content/best.pt')
shutil.make_archive('/content/train_artifacts', 'zip', 'runs/yolov8s_ppe')
files.download('/content/best.pt')
files.download('/content/train_artifacts.zip')  # results.png, args.yaml, kurva loss/mAP

# (opsional) salin ke Drive juga:
# shutil.copy('/content/best.pt', '/content/drive/MyDrive/best.pt')